In [25]:
import pandas as pd

import gzip
import xml.etree.ElementTree as ET

In [26]:
output_folder = "./simulation_data_filtered"
input_folder = "./Neuchatel_10pct"

# Filter output_persons

In [27]:
person_path = input_folder + "/simulation_output/output_persons.csv.gz"
person_df = pd.read_csv(person_path, compression='gzip', low_memory=False, sep=";")

print(person_df.columns)
person_df_filter = person_df[person_df['cantonName'] == 'neuchatel']

person_df_filter.to_csv(output_folder + "/output_persons_neuchatel.csv.gz", index=False, compression='gzip', sep=";")

print("Number of persons in all cantons: ", len(person_df))
print("Number of persons in Neuchatel: ", len(person_df_filter))

Index(['person', 'executed_score', 'first_act_x', 'first_act_y',
       'first_act_type', 'age', 'bikeAvail', 'cantonId', 'cantonName',
       'carASC', 'carAvail', 'carOwnershipRatio', 'employed',
       'hasBikeLoopTrip', 'hasCarLoopTrip', 'hasCarPassengerLoopTrip',
       'hasLicense', 'hasPtLoopTrip', 'hasWalkLoopTrip', 'home_coordinate_x',
       'home_coordinate_y', 'home_x', 'home_y', 'incomePerCapita',
       'isCarPassenger', 'isCrossBorder', 'isExternalFR', 'isFreight',
       'isOutside', 'municipalityType', 'numberOfCars', 'ovgk', 'sex',
       'spRegion', 'subscriptions', 'crossBorderOD', 'subpopulation', 'type'],
      dtype='str')
Number of persons in all cantons:  28160
Number of persons in Neuchatel:  17774


# Filter neuchatel_population


In [28]:
input_path = input_folder + "/neuchatel_population.xml.gz"
output_path = output_folder + "/neuchatel_population_residents.xml.gz"

header = (
    '<?xml version="1.0" encoding="utf-8"?>\n'
    '<!DOCTYPE population SYSTEM "http://www.matsim.org/files/dtd/population_v6.dtd">\n'
    '<population desc="Neuchatel Residents Only">\n'
    '\t<attributes>\n'
    '\t\t<attribute name="coordinateReferenceSystem" class="java.lang.String">Atlantis</attribute>\n'
    '\t</attributes>\n\n'
)

count = 0
with gzip.open(input_path, 'rb') as fin, gzip.open(output_path, 'wb') as fout:
    fout.write(header.encode('utf-8'))
    
    # Stream through agents one by one
    for event, elem in ET.iterparse(fin, events=('end',)):
        if elem.tag == 'person':
            is_neuchatel = False
            for attr in elem.findall('./attributes/attribute'):
                if attr.get('name') == 'cantonName' and attr.text == 'neuchatel':
                    is_neuchatel = True
                    break
            
            if is_neuchatel:
                count += 1
                fout.write(b'\t' + ET.tostring(elem, encoding='utf-8') + b'\n')
            
            # Clear element from memory
            elem.clear()
            
    fout.write(b'</population>\n')
print(f"Filtered {count} persons living in Neuchâtel saved to {output_path}")

Filtered 17774 persons living in Neuchâtel saved to ./simulation_data_filtered/neuchatel_population_residents.xml.gz
